#Ingestion Techniques to Bronze

###1. CTAS (Create Table As Select)
- CTAS stands for Create Table As Select.
- Used to create a new table from an existing query or dataset.
- Data is loaded into the target table while creating it.
- Commonly used for **batch ingestion**.



![](/Workspace/Users/rinkimehra068@gmail.com/Multihop Demo/batch_ing.JPG)

In [0]:
CREATE TABLE IF NOT EXISTS dev.bronze.sales AS
SELECT *,
        _metadata.file_name as _source_name,
        _metadata.file_path as _source_path,
        current_timestamp() as _ingested_at
FROM read_files("/Volumes/dev/bronze/raw/sales/sales.csv");

In [0]:
select count(*) from dev.bronze.sales

###2. COPY INTO
- SQL command used to load files into Delta tables.
- Supports formats like CSV, JSON, Parquet and Avro.
- Tracks previously loaded files.
- Prevents re-ingestion of already loaded files.
- Supports incremental batch ingestion.
- Supports schema evolution with appropriate configuration.



![](/Workspace/Users/rinkimehra068@gmail.com/Multihop Demo/incremental.JPG)

In [0]:
CREATE TABLE IF NOT EXISTS dev.bronze.sales2;

COPY INTO dev.bronze.sales2
FROM '/Volumes/dev/bronze/raw/sales/'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

###3. Auto Loader

- Databricks feature for incremental file ingestion.
- Uses cloudFiles format.
- Supports cloud storage and Unity Catalog Volumes.
- Automatically detects new files.
- Supports schema inference and schema evolution.
- Uses checkpoints to track processed files.
- Supports scalable file ingestion using Structured Streaming.



![](/Workspace/Users/rinkimehra068@gmail.com/Multihop Demo/streaming.JPG)

In [0]:
%python
from pyspark.sql.functions import col, current_timestamp

source_path = "/Volumes/dev/bronze/raw/sales/"
schema_location = "/Volumes/dev/bronze/raw/sales/_schemas/sales3"

(
  spark.readStream
       .format("cloudFiles")
       .option("cloudFiles.format", "csv")
       .option("header", "true")
       .option("cloudFiles.schemaLocation", schema_location)
       .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
       .load(source_path)
       .withColumn("_source_name", col("_metadata.file_name"))
       .withColumn("_source_path", col("_metadata.file_path"))
       .withColumn("_ingested_at", current_timestamp())
       .writeStream
       .option("checkpointLocation", f"{schema_location}/_checkpoint")
       .trigger(availableNow=True)
       .toTable("dev.bronze.sales3")
)

In [0]:
select count(*) from dev.bronze.sales3